# **ResNet50 — EXP02 CBAM (Convolutional Block Attention Module)**

Versi CBAM dari `exp01-cnn-baseline.ipynb` (ResNet50 Baseline). Mengikuti prinsip isolasi yang sama seperti versi ECA: **SEMUA hyperparameter identik dengan baseline**, satu-satunya perubahan adalah penyisipan modul CBAM -- supaya efek CBAM terhadap performa bisa diatribusikan murni ke modul attention-nya, bukan ke confound lain.

**Perbedaan CBAM vs ECA:**
- ECA cuma channel attention (1D conv atas hasil avg-pool per channel).
- CBAM (Woo et al., 2018) punya **dua submodul sequential**: (1) *Channel Attention* -- shared MLP atas avg-pool DAN max-pool per channel, digabung lalu di-sigmoid; (2) *Spatial Attention* -- avg-pool & max-pool sepanjang axis channel, digabung jadi 2 channel, lalu conv 7x7 + sigmoid. Channel attention diterapkan dulu, baru spatial attention terhadap hasilnya (`x' = Mc(x) ⊗ x`, lalu `x'' = Ms(x') ⊗ x'`).

**Titik insersi CBAM (sama seperti ECA):**
- ResNet punya banyak feature map 4D di sepanjang backbone -- CBAM disisipkan di slot `.se` yang memang disediakan timm di setiap **Bottleneck block** (persis setelah `conv3`+`bn3`, sebelum residual add), sesuai desain CBAM asli yang memasang di seluruh backbone CNN, bukan cuma 1 titik.


## 1. Import & Setup

In [1]:
# 1. Install & Import
import os, copy, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# PERBAIKAN (dari ViT exp01): AMP -- sebagian besar operasi forward jalan di
# float16 (lebih cepat & hemat VRAM di GPU RTX/Tensor Core), backward/update tetap
# presisi lewat GradScaler. Belum ada di kedua notebook ResNet sebelumnya.
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms, datasets
from PIL import Image
from tqdm import tqdm

import timm   # PERBAIKAN: torchvision.models -> timm, biar 1 API dipakai semua arsitektur (ResNet18/50, EfficientNet, ViT, dst)
import wandb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Ambil API key dari environment variable
wandb_api_key = os.getenv("WANDB_API_KEY")

os.environ["TORCH_HOME"] = "D:/cache/torch"
os.environ["HF_HOME"] = "D:/cache/huggingface"

os.environ["WANDB_DIR"] = "D:/cache/wandb"
os.environ["WANDB_CACHE_DIR"] = "D:/cache/wandb_cache"

os.environ["TEMP"] = "D:/cache/temp"
os.environ["TMP"] = "D:/cache/temp"

os.environ["CUDA_CACHE_PATH"] = "D:/cache/cuda"

print(os.getcwd())

# Login ke wandb
wandb.login(key=wandb_api_key)

# PERBAIKAN (dari ViT exp01 + ResNet18): seed eksplisit -- EXP01-03 ResNet50 lama
# cuma nge-seed StratifiedKFold, TIDAK nge-seed init bobot FC head / urutan shuffle.
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# PERBAIKAN (dari ViT exp01): cudnn.benchmark auto-tune algoritma konvolusi
# tercepat untuk ukuran input yang konsisten (semua di-resize ke 224x224).
torch.backends.cudnn.benchmark = True


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\UNIDA\_netrc.


d:\Devianest_SkripsiTest\Code_CNN


wandb: Currently logged in as: devianestnarendra to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


## 2. Config

In [2]:
TRAIN_DIR = r"D:\Devianest_SkripsiTest\train"
TEST_DIR  = r"D:\Devianest_SkripsiTest\test"

# PERBAIKAN (VSCode lokal): ganti "/kaggle/working" -> folder "outputs" relatif
# terhadap lokasi notebook ini. os.makedirs(..., exist_ok=True) otomatis bikin
# foldernya kalau belum ada, supaya tidak error "No such file or directory"
# saat pertama kali disimpan.
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ARCH_KEY  = "EXP04_ResNet50_CBAM"
TIMM_NAME = "resnet50"

IMG_SIZE     = 224
BATCH_SIZE   = 32          # tetap seperti EXP01-03 (ResNet50 lebih berat dari ResNet18, batch lebih kecil)
EPOCHS       = 50
N_FOLDS      = 5
DROPOUT      = 0.3         # dipertahankan dari EXP01-03 (nilai yang sudah teruji utk ResNet50)
LR           = 1e-4        # PERBAIKAN: dipertahankan nilai ResNet (BUKAN LR ViT 3e-5) -- CNN historically
                            # lebih toleran ke LR sedikit lebih tinggi dibanding attention layer ViT yang sensitif
WEIGHT_DECAY = 1e-4         # dipertahankan nilai konvensi CNN transfer learning (bukan WD ViT 0.01)
LABEL_SMOOTHING = 0.1
EARLY_STOP_PATIENCE = 7     # PERBAIKAN: naik dari 5 -> 7 (lebih toleran sebelum berhenti)

WANDB_PROJECT = "SkinDisease-CNN"   # disamakan dengan notebook ResNet18, biar 1 project W&B

# ResNet50: unfreeze layer2/3/4 + fc -- sama scope kapasitas dengan EXP03 (untuk
# tetap bisa dibandingkan), TAPI training regime-nya sudah diperbaiki (lihat bawah).
UNFREEZE_PATTERNS = ["layer2", "layer3", "layer4", "fc"]

# PERBAIKAN (dari ViT exp01 + ResNet18): scheduler ReduceLROnPlateau disamakan
# PERSIS dengan setting yang sudah dipakai di ViT & ResNet18 -- root cause
# instabilitas ResNet50 lama adalah OneCycleLR yang di-set untuk siklus 50 epoch
# penuh, tapi EarlyStopping hampir selalu memotong training di epoch 11-22
# (SEBELUM fase anneal selesai) -- lihat diskusi sebelumnya soal Fold 4 yang
# selalu menang karena satu-satunya fold yang tidak pernah early-stop.
SCHEDULER_FACTOR    = 0.1
SCHEDULER_PATIENCE  = 2
SCHEDULER_THRESHOLD = 1e-4
SCHEDULER_MIN_LR    = 1e-7


# ── CBAM (Convolutional Block Attention Module) ──────────────────────────────
# PERBAIKAN: versi CBAM dari EXP01 -- SEMUA hyperparameter di atas dipertahankan
# PERSIS SAMA (LR, WEIGHT_DECAY, DROPOUT, UNFREEZE_PATTERNS, scheduler, dst) supaya
# efek CBAM terisolasi (tidak ada confound lain), sama seperti prinsip versi ECA.
USE_CBAM               = True
CBAM_REDUCTION_RATIO   = 16   # rasio reduksi shared MLP channel attention, default paper CBAM (Woo et al., 2018)
CBAM_SPATIAL_KERNEL_SIZE = 7  # ukuran kernel conv2d spatial attention, default paper CBAM


## 3. Dataset & Augmentasi

In [3]:
# PERBAIKAN: augmentasi disamakan PERSIS dengan versi terbaru ViT exp01 (medium,
# termasuk RandomResizedCrop) -- bukan versi ResNet18 yang belum pakai
# RandomResizedCrop. Ini penting supaya CNN dan ViT dibandingkan dengan
# preprocessing yang identik (bukan confound tambahan).
def get_transforms(img_size):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf


classes = sorted(os.listdir(TRAIN_DIR))
class_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)

filepaths, labels = [], []
for label in classes:
    class_path = os.path.join(TRAIN_DIR, label)
    for img in os.listdir(class_path):
        filepaths.append(os.path.join(class_path, img))
        labels.append(class_to_idx[label])

print("Total Images :", len(filepaths))
print("Classes      :", num_classes)


class SkinDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


Total Images : 15557
Classes      : 23


## 4. Early Stopping (val_f1) + K-Fold

In [4]:
class EarlyStopping:
    # Kriteria val_f1 (bukan val_loss) -- konsisten dengan ViT exp01 & ResNet18:
    # lebih robust untuk data imbalanced (23 kelas DermNet) dibanding val_loss.
    def __init__(self, patience=5):
        self.patience = patience
        self.best_f1 = -np.inf
        self.counter = 0

    def step(self, val_f1):
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


## 5. Model Builder + Freeze Strategy

In [5]:
# ── CBAM (Convolutional Block Attention Module) ──────────────────────────────
# Sama seperti versi ECA, titik insersinya identik karena arsitekturnya sama:
# - ResNet (CNN) punya feature map 4D (B, C, H, W) di SEPANJANG backbone -- setiap
#   Bottleneck block. timm sudah menyediakan slot khusus untuk ini: atribut
#   `.se` di tiap Bottleneck (dipakai bawaan buat SE-Net-style attention),
#   dipanggil PERSIS setelah conv3+bn3, SEBELUM residual add:
#       x = self.conv3(x); x = self.bn3(x)
#       if self.se is not None: x = self.se(x)
#       ...; x += shortcut; x = self.act3(x)
#   Bedanya dengan ECA: CBAM adalah 2 submodul SEQUENTIAL -- channel attention
#   dulu (Mc), baru spatial attention (Ms) terhadap hasil channel attention.
class ChannelAttention(nn.Module):
    # Channel attention CBAM: shared MLP (bottleneck 1x1 conv) diterapkan ke
    # HASIL avg-pool DAN max-pool secara terpisah, lalu dijumlahkan -- beda
    # dengan SE-Net/ECA yang cuma pakai avg-pool saja. Menambahkan max-pool
    # menangkap sinyal channel yang "menonjol", bukan cuma rata-rata.
    def __init__(self, channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(channels // reduction_ratio, 1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, H, W)
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        y = self.sigmoid(avg_out + max_out)
        return x * y


class SpatialAttention(nn.Module):
    # Spatial attention CBAM: pool sepanjang axis CHANNEL (bukan spatial) --
    # avg & max per posisi (h, w) di-concat jadi 2-channel map, lalu conv 7x7
    # (kernel besar supaya konteks spasial lebih luas) + sigmoid.
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=(kernel_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, H, W) -- sudah hasil channel attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)   # (B, 2, H, W)
        y = self.conv(y)
        y = self.sigmoid(y)
        return x * y


class CBAMAttention(nn.Module):
    # Gabungan sequential: channel attention dulu, baru spatial attention
    # terhadap hasilnya -- sesuai urutan channel-first yang terbukti lebih
    # baik di paper CBAM asli (Woo et al., 2018) dibanding urutan sebaliknya.
    def __init__(self, channels, reduction_ratio=16, spatial_kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction_ratio)
        self.spatial_attention = SpatialAttention(spatial_kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


def inject_cbam(model, reduction_ratio=16, spatial_kernel_size=7):
    # PERBAIKAN: CBAM disisipkan ke SEMUA Bottleneck block (layer1-4), bukan cuma
    # layer yang di-unfreeze -- sesuai desain CBAM asli yang memasang attention
    # di seluruh backbone. Scope freeze/unfreeze BACKBONE tetap diatur terpisah
    # lewat apply_freeze_strategy (lihat PERBAIKAN di bawah: modul CBAM sendiri
    # selalu trainable, terlepas dari block induknya beku atau tidak).
    n_injected = 0
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name, None)
        if layer is None:
            continue
        for block in layer:
            channels = block.bn3.num_features
            block.se = CBAMAttention(channels, reduction_ratio=reduction_ratio, spatial_kernel_size=spatial_kernel_size)
            n_injected += 1
    print(f"  [CBAM] Disisipkan ke {n_injected} Bottleneck block (layer1-4).")
    return model


In [6]:
def build_model(num_classes, dropout=DROPOUT):
    # PERBAIKAN: head sederhana bawaan timm (Linear + drop_rate), BUKAN custom
    # Linear->BN->ReLU->Dropout->Linear seperti EXP01-03 lama. Diselaraskan dengan
    # ViT exp01 & ResNet18 -- supaya kapasitas head TIDAK jadi confound tambahan
    # saat membandingkan arsitektur (kalau satu arsitektur dikasih head lebih besar
    # dari yang lain, selisih performa bisa jadi cuma soal head, bukan backbone).
    model = timm.create_model(
        TIMM_NAME, pretrained=True, num_classes=num_classes, drop_rate=dropout
    )
    # PERBAIKAN (CBAM): sisipkan CBAM ke tiap Bottleneck SETELAH model pretrained
    # dibuat -- supaya bobot pretrained conv1/2/3/bn tidak tersentuh, cuma nambah
    # modul baru di slot `.se` yang random-init.
    if USE_CBAM:
        model = inject_cbam(model, reduction_ratio=CBAM_REDUCTION_RATIO, spatial_kernel_size=CBAM_SPATIAL_KERNEL_SIZE)
    return model


def apply_freeze_strategy(model, patterns=UNFREEZE_PATTERNS):
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(pat in name for pat in patterns):
            p.requires_grad = True

    # PERBAIKAN (CBAM): modul CBAM SELALU trainable, terlepas dari UNFREEZE_PATTERNS
    # -- konsisten dengan versi ECA (attention submodule selalu unfrozen).
    # Bobotnya random-init (bukan pretrained), jadi kalau block induknya kebetulan
    # di luar scope unfreeze (mis. layer1), CBAM di block itu tidak boleh ikut beku,
    # kalau tidak dia jadi dead weight (random & tidak pernah di-update).
    if USE_CBAM:
        for module in model.modules():
            if isinstance(module, CBAMAttention):
                for p in module.parameters():
                    p.requires_grad = True

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  [{ARCH_KEY}] Trainable params: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")
    return model


## 6. Train 1 Fold

In [7]:
def train_one_fold(fold, train_idx, val_idx, run):
    train_tf, eval_tf = get_transforms(IMG_SIZE)

    train_files  = [filepaths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_files    = [filepaths[i] for i in val_idx]
    val_labels   = [labels[i] for i in val_idx]

    # PERBAIKAN (dari ViT exp01): pin_memory=True -- percepat transfer CPU->GPU.
    # num_workers=2 dari ResNet18 (lebih cepat load data daripada 0 di EXP03 lama).
    train_loader = DataLoader(
        SkinDataset(train_files, train_labels, train_tf),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        SkinDataset(val_files, val_labels, eval_tf),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True
    )

    # PERBAIKAN: reseed per fold -- inisialisasi FC head & urutan shuffle
    # reproducible, tidak tergantung urutan eksekusi fold sebelumnya.
    random.seed(SEED + fold); np.random.seed(SEED + fold)
    torch.manual_seed(SEED + fold); torch.cuda.manual_seed_all(SEED + fold)

    model = build_model(num_classes)
    model = apply_freeze_strategy(model)
    model = model.to(device)

    # PERBAIKAN (dari ViT exp01): class_weights dinormalisasi supaya rata-rata = 1.
    # EXP01-03 & ResNet18 sebelumnya cuma 1./bincount TANPA normalisasi --
    # magnitude weight antar kelas timpang, jadi salah satu sumber loss yang
    # "melompat" tergantung komposisi kelas tiap batch.
    class_counts  = np.bincount(train_labels, minlength=num_classes)
    class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * num_classes

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device), label_smoothing=LABEL_SMOOTHING
    )
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY
    )
    # PERBAIKAN: scheduler disamakan persis dengan ViT exp01 & ResNet18 -- root
    # cause instabilitas ResNet50 lama (OneCycleLR + EarlyStopping yang memotong
    # sebelum siklus selesai) sudah tidak ada lagi di sini.
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE,
        threshold=SCHEDULER_THRESHOLD, min_lr=SCHEDULER_MIN_LR
    )

    # PERBAIKAN (dari ViT exp01): AMP -- autocast di forward pass, GradScaler
    # untuk backward/update supaya gradient float16 tidak underflow.
    scaler = GradScaler()

    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)
    best_val_f1 = -np.inf
    # PERBAIKAN: tracking accuracy/precision/recall di titik checkpoint terbaik juga
    # (sebelumnya cuma f1) -- supaya format output/summary sama seperti ViT exp01,
    # yang melaporkan keempat metrik (bukan cuma F1) di rekap akhir.
    best_val_acc = -np.inf
    best_val_precision = -np.inf
    best_val_recall = -np.inf
    best_val_loss = np.inf
    best_train_loss = np.inf
    best_model_path = None
    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        print(f"\n[{ARCH_KEY} | fold {fold+1}] Epoch {epoch+1}/{EPOCHS} (LR: {optimizer.param_groups[0]['lr']:.2e})")

        # TRAIN
        model.train()
        train_loss = 0
        for imgs, tgts in tqdm(train_loader, desc="Train"):
            imgs, tgts = imgs.to(device), tgts.to(device)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), tgts)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # VALIDATION
        model.eval()
        val_loss = 0
        preds, trues = [], []
        with torch.no_grad():
            for imgs, tgts in tqdm(val_loader, desc="Val"):
                imgs, tgts = imgs.to(device), tgts.to(device)
                with autocast():
                    outputs = model(imgs)
                    v_loss  = criterion(outputs, tgts)
                val_loss += v_loss.item()
                preds.extend(outputs.argmax(1).cpu().numpy())
                trues.extend(tgts.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)

        acc       = accuracy_score(trues, preds)
        precision = precision_score(trues, preds, average="weighted", zero_division=0)
        recall    = recall_score(trues, preds, average="weighted", zero_division=0)
        f1        = f1_score(trues, preds, average="weighted", zero_division=0)

        scheduler.step(f1)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(f"Train Loss : {avg_train_loss:.4f} | Val Loss  : {avg_val_loss:.4f}")
        print(f"Accuracy   : {acc:.4f}  | Precision : {precision:.4f}")
        print(f"Recall     : {recall:.4f}  | F1 Score  : {f1:.4f}")

        run.log({
            "epoch": epoch + 1,
            f"fold_{fold+1}/train_loss": avg_train_loss,
            f"fold_{fold+1}/val_loss": avg_val_loss,
            f"fold_{fold+1}/accuracy": acc,
            f"fold_{fold+1}/precision": precision,
            f"fold_{fold+1}/recall": recall,
            f"fold_{fold+1}/f1_score": f1,
            f"fold_{fold+1}/lr": optimizer.param_groups[0]["lr"],
        })

        # SAVE BEST MODEL -- kriteria val_f1 tertinggi (bukan val_loss terendah)
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_acc = acc
            best_val_precision = precision
            best_val_recall = recall
            best_val_loss = avg_val_loss
            best_train_loss = avg_train_loss

            save_path = f"{OUTPUT_DIR}/{ARCH_KEY}_fold{fold+1}.pth"
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_loss": avg_val_loss,
                "f1": f1,
                "fold": fold + 1,
                "arch": ARCH_KEY,
            }, save_path)
            best_model_path = save_path
            print(f"  ✓ Model saved → {save_path} (F1: {f1:.4f})")

        if early_stopping.step(f1):
            print("Early Stopping Triggered")
            break

    # ── PLOT LOSS CURVE PER FOLD ──────────────────────────────────────────────
    epochs_ran = range(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs_ran, train_losses, label="Train Loss", marker="o", markersize=3)
    ax.plot(epochs_ran, val_losses, label="Val Loss", marker="o", markersize=3)
    ax.set_title(f"{ARCH_KEY} — Fold {fold+1} Loss Curve")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    curve_path = f"{OUTPUT_DIR}/{ARCH_KEY}_Fold_{fold+1}_Loss_Curve.png"
    fig.savefig(curve_path, dpi=150, bbox_inches="tight")
    run.log({f"Loss_Curve/Fold_{fold+1}": wandb.Image(curve_path)})
    plt.close(fig)

    return {
        "arch": ARCH_KEY,
        "fold": fold + 1,
        "train_loss": best_train_loss,
        "val_loss": best_val_loss,
        # PERBAIKAN: sertakan accuracy/precision/recall (bukan cuma f1), supaya
        # results_df punya kolom yang sama seperti fold_accuracies/fold_precision/
        # fold_recall/fold_f1 di ViT exp01.
        "accuracy": best_val_acc,
        "precision": best_val_precision,
        "recall": best_val_recall,
        "f1": best_val_f1,
        "model_path": best_model_path,
    }


## 7. MAIN LOOP — 5 Fold (ResNet50)

Kalau waktu habis di tengah jalan: checkpoint tiap fold udah ke-save duluan (di `all_results`), aman buat dilanjut manual per-fold.

In [8]:
all_results = []

run = wandb.init(
    project="SkinDisease-CNN",
    entity="devianestnarendra_Team",
    name=f"{ARCH_KEY}",
    reinit=True,
    config={
        "architecture": ARCH_KEY,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "img_size": IMG_SIZE,
        "optimizer": "AdamW",
        "scheduler": f"ReduceLROnPlateau(mode=max, factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})",
        "lr": LR,
        "dropout": DROPOUT,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "unfreeze_patterns": UNFREEZE_PATTERNS,
        "checkpoint_criteria": "best_val_f1",
        "amp": True,
        "seed": SEED,
        "use_cbam": USE_CBAM,
        "cbam_reduction_ratio": CBAM_REDUCTION_RATIO,
        "cbam_spatial_kernel_size": CBAM_SPATIAL_KERNEL_SIZE,
    }
)

for fold, (train_idx, val_idx) in enumerate(skf.split(filepaths, labels)):
    result = train_one_fold(fold, train_idx, val_idx, run)
    all_results.append(result)

run.finish()

results_df = pd.DataFrame(all_results)
results_df


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3272042763.py:54: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  [CBAM] Disisipkan ke 16 Bottleneck block (layer1-4).
  [EXP04_ResNet50_CBAM] Trainable params: 25,846,327 / 26,071,671 (99.1%)

[EXP04_ResNet50_CBAM | fold 1] Epoch 1/50 (LR: 1.00e-04)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3272042763.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3272042763.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:48<00:00,  2.01it/s]


Train Loss : 3.1606 | Val Loss  : 3.1581
Accuracy   : 0.1899  | Precision : 0.2913
Recall     : 0.1899  | F1 Score  : 0.1633
  ✓ Model saved → outputs/EXP04_ResNet50_CBAM_fold1.pth (F1: 0.1633)

[EXP04_ResNet50_CBAM | fold 1] Epoch 2/50 (LR: 1.00e-04)


...

Train Loss : 1.1302 | Val Loss  : 2.2643
Accuracy   : 0.5648  | Precision : 0.5802
Recall     : 0.5648  | F1 Score  : 0.5658

[EXP04_ResNet50_CBAM | fold 5] Epoch 43/50 (LR: 1.00e-07)


Train:   0%|          | 0/389 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3272042763.py:78: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val:   0%|          | 0/98 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3272042763.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Val: 100%|██████████| 98/98 [00:26<00:00,  3.65it/s]


Train Loss : 1.1307 | Val Loss  : 2.2601
Accuracy   : 0.5696  | Precision : 0.5825
Recall     : 0.5696  | F1 Score  : 0.5706
Early Stopping Triggered


epoch,▃▄▄▄▅▆▇▇▇█▁▁▃▃▃▅▆▆▆▇█▁▁▂▂▅▆▆▇█▁▂▂▃▄▇▇▂▂▇
fold_1/accuracy,▁▂▃▃▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████
fold_1/f1_score,▁▂▃▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇█████████████
fold_1/lr,████████████████████████████████▂▂▂▂▂▂▂▁
fold_1/precision,▁▂▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇█████▇███████
fold_1/recall,▁▂▃▃▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████
fold_1/train_loss,█▇▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_1/val_loss,█▆▅▅▅▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
fold_2/accuracy,▁▂▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██████████████████
fold_2/f1_score,▁▂▃▃▄▅▅▅▅▆▆▆▇▆▇▇▇▇▇█▇▇██████████████████
+26,...


,arch,fold,train_loss,val_loss,accuracy,precision,recall,f1,model_path
0,EXP04_ResNet50_CBAM,1,1.003737,2.203251,0.589974,0.594686,0.589974,0.589395,outputs/EXP04_ResNet50_CBAM_fold1.pth
1,EXP04_ResNet50_CBAM,2,1.153994,2.256412,0.560090,0.572952,0.560090,0.561720,outputs/EXP04_ResNet50_CBAM_fold2.pth
2,EXP04_ResNet50_CBAM,3,1.058845,2.287378,0.561877,0.571653,0.561877,0.563868,outputs/EXP04_ResNet50_CBAM_fold3.pth
3,EXP04_ResNet50_CBAM,4,1.008970,2.212225,0.600129,0.609496,0.600129,0.600760,outputs/EXP04_ResNet50_CBAM_fold4.pth
4,EXP04_ResNet50_CBAM,5,1.144472,2.249579,0.575699,0.588896,0.575699,0.577367,outputs/EXP04_ResNet50_CBAM_fold5.pth


## 8. Rekap 5-Fold

In [9]:
# PERBAIKAN: format summary disamakan persis dengan ViT exp01 -- laporkan
# Mean ± Std untuk keempat metrik (Accuracy, Precision, Recall, F1), bukan
# cuma F1 saja.
print("\n" + "="*50)
print(f"  FINAL RESULT — ALL FOLDS ({ARCH_KEY})")
print("="*50)
print(f"Mean Accuracy  : {results_df['accuracy'].mean():.4f} ± {results_df['accuracy'].std():.4f}")
print(f"Mean Precision : {results_df['precision'].mean():.4f} ± {results_df['precision'].std():.4f}")
print(f"Mean Recall    : {results_df['recall'].mean():.4f} ± {results_df['recall'].std():.4f}")
print(f"Mean F1 Score  : {results_df['f1'].mean():.4f} ± {results_df['f1'].std():.4f}")

# ── WANDB LOG SUMMARY (format sama seperti ViT exp01) ─────────────────────────
try:
    run.log({
        "summary/mean_accuracy"  : results_df["accuracy"].mean(),
        "summary/mean_precision" : results_df["precision"].mean(),
        "summary/mean_recall"    : results_df["recall"].mean(),
        "summary/mean_f1"        : results_df["f1"].mean(),
        "summary/std_accuracy"   : results_df["accuracy"].std(),
        "summary/std_precision"  : results_df["precision"].std(),
        "summary/std_recall"     : results_df["recall"].std(),
        "summary/std_f1"         : results_df["f1"].std(),
    })
except Exception as e:
    print(f"  ⚠ W&B log summary gagal (dilewati): {e}")

# ── GRAFIK: 4 metrik per fold (bukan cuma F1) ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
metric_cols = ["accuracy", "precision", "recall", "f1"]
metric_titles = ["Accuracy", "Precision", "Recall", "F1-Score"]

for ax, col, title in zip(axes, metric_cols, metric_titles):
    ax.bar(results_df["fold"].astype(str), results_df[col], color="#1f3a5f")
    ax.axhline(results_df[col].mean(), color="red", linestyle="--", label=f"Mean = {results_df[col].mean():.3f}")
    ax.set_xlabel("Fold"); ax.set_ylabel(f"Val {title}")
    ax.set_title(f"{ARCH_KEY} — {title} per Fold")
    ax.set_ylim(0, 1); ax.legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_fold_metrics_chart.png", dpi=200)
plt.show()

results_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_all_folds.csv", index=False)



  FINAL RESULT — ALL FOLDS (EXP04_ResNet50_CBAM)
Mean Accuracy  : 0.5776 ± 0.0175
Mean Precision : 0.5875 ± 0.0158
Mean Recall    : 0.5776 ± 0.0175
Mean F1 Score  : 0.5786 ± 0.0167
  ⚠ W&B log summary gagal (dilewati): Run (7quzcmr1) is finished. The call to `log` will be ignored. Please make sure that you are using an active run.


C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\1248490208.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Test Evaluation (Fold Terbaik)

In [10]:
best_fold_result = max(all_results, key=lambda r: r["f1"])
best_overall_path = best_fold_result["model_path"]
print(f"Fold terbaik    : {best_fold_result['fold']}")
print(f"Checkpoint      : {best_overall_path}")
print(f"Val F1 terbaik  : {best_fold_result['f1']:.4f}")

_, eval_tf = get_transforms(IMG_SIZE)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = build_model(num_classes)
model = model.to(device)
checkpoint = torch.load(best_overall_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, tgts in tqdm(test_loader, desc="Test"):
        imgs = imgs.to(device)
        with autocast():
            out = model(imgs)
        y_true.extend(tgts.numpy())
        y_pred.extend(out.argmax(1).cpu().numpy())

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(15, 15))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(cmap="Blues", ax=ax, xticks_rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Confusion_Matrix.png", dpi=150, bbox_inches="tight")
plt.show()

test_summary_df = pd.DataFrame([{
    "arch": ARCH_KEY, "Accuracy": acc, "Precision": precision, "Recall": recall, "F1": f1
}])
test_summary_df.to_csv(f"{OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv", index=False)
print(f"\n✓ Test summary disimpan -> {OUTPUT_DIR}/{ARCH_KEY}_Test_Summary.csv")


Fold terbaik    : 4
Checkpoint      : outputs/EXP04_ResNet50_CBAM_fold4.pth
Val F1 terbaik  : 0.6008
  [CBAM] Disisipkan ke 16 Bottleneck block (layer1-4).


Test:   0%|          | 0/126 [00:00<?, ?it/s]C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3000596476.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Test: 100%|██████████| 126/126 [00:53<00:00,  2.33it/s]


Accuracy  : 0.6072
Precision : 0.6202
Recall    : 0.6072
F1-Score  : 0.6083

                                                                    precision    recall  f1-score   support

                                           Acne and Rosacea Photos       0.77      0.90      0.83       312
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions       0.70      0.59      0.64       288
                                          Atopic Dermatitis Photos       0.50      0.58      0.54       123
                                            Bullous Disease Photos       0.54      0.48      0.51       113
                Cellulitis Impetigo and other Bacterial Infections       0.32      0.51      0.39        73
                                                     Eczema Photos       0.57      0.66      0.61       309
                                      Exanthems and Drug Eruptions       0.42      0.59      0.49       101
                 Hair Loss Photos Alopecia and other Hair 

C:\Users\UNIDA\AppData\Local\Temp\ipykernel_15268\3000596476.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9b. Per-Image Test Prediction (True/False per gambar)

Dump terpisah dari summary agregat di atas -- CSV ini berisi 1 baris per gambar test (filepath, label asli, label prediksi, benar/salah), supaya bisa dicek manual gambar mana saja yang salah diklasifikasikan (misal untuk lampiran/analisis kualitatif di skripsi).

In [11]:
# PERBAIKAN: dump prediksi per-gambar (bukan cuma summary agregat) -- pakai
# y_true/y_pred/test_dataset yang sudah dihitung di cell sebelumnya (state Jupyter
# masih ada, tidak perlu re-run inference).
test_filepaths = [fp for fp, _ in test_dataset.samples]   # urutan sama dgn y_true/y_pred (shuffle=False)
assert len(test_filepaths) == len(y_true) == len(y_pred), "Jumlah filepath tidak sama dengan jumlah prediksi!"

per_image_df = pd.DataFrame({
    "filepath": test_filepaths,
    "true_label": [classes[t] for t in y_true],
    "pred_label": [classes[p] for p in y_pred],
    "correct": [t == p for t, p in zip(y_true, y_pred)],
})

per_image_csv_path = f"{OUTPUT_DIR}/{ARCH_KEY}_Test_PerImage_Predictions.csv"
per_image_df.to_csv(per_image_csv_path, index=False)

n_correct = int(per_image_df["correct"].sum())
n_total = len(per_image_df)
print(f"✓ Per-image prediction disimpan -> {per_image_csv_path}")
print(f"  Benar : {n_correct} / {n_total} ({100 * n_correct / n_total:.2f}%)")
print(f"  Salah : {n_total - n_correct} / {n_total} ({100 * (n_total - n_correct) / n_total:.2f}%)")

# Preview baris yang salah -- 10 contoh pertama, berguna buat dicek manual
per_image_df[~per_image_df["correct"]].head(10)


✓ Per-image prediction disimpan -> outputs/EXP04_ResNet50_CBAM_Test_PerImage_Predictions.csv
  Benar : 2430 / 4002 (60.72%)
  Salah : 1572 / 4002 (39.28%)


,filepath,true_label,pred_label,correct
5,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Tinea Ringworm Candidiasis and other Fungal In...,False
14,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Melanoma Skin Cancer Nevi and Moles,False
17,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Warts Molluscum and other Viral Infections,False
55,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Atopic Dermatitis Photos,False
66,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Tinea Ringworm Candidiasis and other Fungal In...,False
73,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Actinic Keratosis Basal Cell Carcinoma and oth...,False
80,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Bullous Disease Photos,False
83,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Exanthems and Drug Eruptions,False
90,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Atopic Dermatitis Photos,False
95,D:\Devianest_SkripsiTest\test\Acne and Rosacea...,Acne and Rosacea Photos,Atopic Dermatitis Photos,False


## Catatan

- **Isolasi variabel**: satu-satunya perbedaan dari `exp01-cnn-baseline.ipynb` adalah `USE_CBAM=True` + penyisipan `CBAMAttention` di tiap Bottleneck (`layer1-4`). Semua hyperparameter lain (LR, WEIGHT_DECAY, DROPOUT, UNFREEZE_PATTERNS, scheduler, augmentasi, seed) identik -- supaya selisih F1 vs EXP01 bisa diatribusikan murni ke CBAM.
- **Titik insersi**: slot `.se` bawaan timm Bottleneck (dipanggil setelah `conv3+bn3`, sebelum residual add) -- bukan hack, ini memang disediakan timm untuk attention module CNN-native (SE-Net dkk), jadi tidak perlu re-write forward pass ResNet secara manual.
- **CBAM = 2 submodul sequential**: channel attention (avg-pool + max-pool -> shared MLP -> sigmoid) diterapkan dulu ke input, baru spatial attention (avg + max sepanjang channel -> conv 7x7 -> sigmoid) diterapkan ke hasilnya. Ini beda dari ECA yang cuma 1 submodul channel attention (tanpa max-pool, tanpa spatial attention).
- **CBAM selalu trainable**: meskipun `layer1` di luar `UNFREEZE_PATTERNS` (backbone-nya beku), modul CBAM di `layer1` tetap dilatih -- karena bobotnya random-init, bukan pretrained. Ini prinsip yang sama dengan versi ECA.
- **Checkpoint & test evaluation**: karena `build_model()` sudah otomatis memanggil `inject_cbam()` saat `USE_CBAM=True`, sel Test Evaluation (§9) TIDAK perlu perubahan apa pun -- `build_model(num_classes)` di situ akan otomatis merekonstruksi arsitektur ber-CBAM sebelum `load_state_dict`, jadi key-nya pasti cocok dengan checkpoint hasil training.
- Checkpoint format & kolom `results_df` tetap sama seperti EXP01 -- tinggal `pd.concat()` untuk rekap akhir baseline vs ECA vs CBAM.
